# einops-reduce composite — cx7: channel-centered image via reduce + right-align broadcast

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-reduce`, `broadcasting-rules`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-reduce"
DD_ATOM_IDS = ["einops-reduce", "broadcasting-rules"]
DD_SUBTOPICS = ["Einops: Reduce", "Numpy: Vectorization and broadcasting"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Centering a feature map along an axis is a two-step move that exercises both atoms in one expression. First `einops.reduce(x, 'b c h w -> b 1 h w', 'mean')` collapses the channel axis to a singleton — the keepdim-style reduce that returns the mean with the reduced axis still present as a size-1 slot. Then `x - mean` triggers NumPy/PyTorch broadcasting: shapes `(B, C, H, W)` and `(B, 1, H, W)` right-align, the size-1 axis is stretched, and the result lands at `x.shape`. Atom 1 controls WHAT shape comes out of the reduction; atom 2 explains WHY subtracting the size-1 tensor doesn't error and lands at the right shape.

### Composite Exercise — channel-centered image via reduce + right-align broadcast

**Atoms exercised together**: `einops-reduce`, `broadcasting-rules`

Implement `cx7_center_per_channel_then_broadcast(x)` that takes a 4-D feature map of shape `(B, C, H, W)` and returns a tensor of the SAME shape, where each `(B, H, W)` slot has been centered against its per-batch per-channel mean over the spatial axes.

Two atoms, one expression:

1. **Reduce with keepdim semantics** — use `einops.reduce(x, 'b c h w -> b c 1 1', 'mean')` (NOT `x.mean(dim=(-2,-1))`). The point is to keep the reduced axes as size-1 slots so the broadcast back works without a manual `unsqueeze`.
2. **Right-align broadcast** — return `x - mean`. The shapes are `(B, C, H, W)` vs `(B, C, 1, 1)`; broadcasting rules right-align them, every pair is either equal or has a 1, so the result is `(B, C, H, W)`. Do not call `.expand` or `.repeat`.

Also assert (inside your fn) that `mean.shape == (B, C, 1, 1)` to make atom 1 visible.

In [ ]:
def cx7_center_per_channel_then_broadcast(x):
    # Atom 1: reduce H,W to size-1 slots (keepdim-style via einops pattern).
    mean = reduce(x, 'b c h w -> b c 1 1', 'mean')
    B, C, _, _ = x.shape
    assert mean.shape == (B, C, 1, 1), mean.shape
    # Atom 2: broadcasting rules — (B,C,H,W) vs (B,C,1,1) right-aligns; size-1 axes stretch.
    return x - mean


<details><summary>Show solution — cx7</summary>

```python
def cx7_center_per_channel_then_broadcast(x):
    # Atom 1: reduce H,W to size-1 slots (keepdim-style via einops pattern).
    mean = reduce(x, 'b c h w -> b c 1 1', 'mean')
    B, C, _, _ = x.shape
    assert mean.shape == (B, C, 1, 1), mean.shape
    # Atom 2: broadcasting rules — (B,C,H,W) vs (B,C,1,1) right-aligns; size-1 axes stretch.
    return x - mean
```

Critically, the pattern `'b c h w -> b c 1 1'` is the einops way of asking for keepdim=True. Without the size-1 placeholders, you'd get `(B, C)` and the subtraction would need an explicit `unsqueeze(-1).unsqueeze(-1)` — losing the composition.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx7',
        'subtopics': ["Einops: Reduce", "Numpy: Vectorization and broadcasting"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()